# Verificación de la escala de coordenadas de Qwen3-VL

El runner convierte `point_2d` a la escala [0, 1000] suponiendo `scale_denominator = 1000`, y su
log marca ese valor como *unconfirmed*. Este notebook dibuja los puntos **ya guardados** sobre las
imágenes que recibió el modelo, sin volver a ejecutar inferencia.

Usa 2 objetos axiales y 2 planares con 6 vistas: el de menor y el de mayor error angular según
`evaluate.py`. Solo lee archivos y escribe los PNG en `CARPETA_SALIDA`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageFilter

RENDERS_ROOT = Path("/home/pespinoz/data/renders")
CARPETA_SALIDA = Path("verificacion_puntos_qwen")
SIZE = 224
LIGHTING = "flat"
N_VIEWS = 6
ARCHIVO_PUNTOS = "qwen3vl_multiview.json"
TOLERANCIA_PX = 4     # margen alrededor de la silueta: los polos pueden caer justo en el borde
UMBRAL_FONDO = 12     # diferencia mínima de color con el fondo para contar como objeto

# Deja None para elegir por error angular, o escribe dos ids: {"axis_sym": ["id1", "id2"], ...}
OBJETOS_MANUALES = None

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

In [ ]:
def elegir_objetos(tipo: str) -> list[tuple[str, float]]:
    """ Elige el objeto de menor y el de mayor error angular con N_VIEWS vistas.

    Args:
        * tipo: 'axis_sym' o 'plane_sym'

    Returns:
        * list[tuple[str, float]]: pares (id del objeto, error angular en grados)

    """
    ruta = RENDERS_ROOT / tipo / f"eval_s{SIZE}_{LIGHTING}_qwen3vl_triangulation_results.json"
    with open(ruta, encoding="utf-8") as archivo:
        objetos = json.load(archivo)["objects"]
    errores = {}
    for obj_id, por_vistas in objetos.items():
        metricas = (por_vistas or {}).get(str(N_VIEWS), {})
        if metricas.get("status") == "ok":
            errores[obj_id] = metricas["angular_error_deg"]
    if OBJETOS_MANUALES is not None:
        return [(obj_id, errores.get(obj_id, float("nan"))) for obj_id in OBJETOS_MANUALES[tipo]]
    ordenados = sorted(errores, key=errores.get)
    return [(ordenados[0], errores[ordenados[0]]), (ordenados[-1], errores[ordenados[-1]])]


def cargar_grupo(tipo: str, obj_id: str) -> tuple[Path, dict]:
    """ Lee el grupo de N_VIEWS vistas del JSON de Qwen3-VL y comprueba que sea modo single.

    Args:
        * tipo: 'axis_sym' o 'plane_sym'
        * obj_id: id del objeto

    Returns:
        * tuple[Path, dict]: carpeta de renders y entrada del JSON

    """
    carpeta = RENDERS_ROOT / tipo / obj_id / str(SIZE) / LIGHTING
    with open(carpeta / ARCHIVO_PUNTOS, encoding="utf-8") as archivo:
        grupo = json.load(archivo)[str(N_VIEWS)]
    if grupo["prompt_mode"] != "single":
        raise ValueError(f"{obj_id}: prompt_mode = {grupo['prompt_mode']}")
    return carpeta, grupo


def mascara_objeto(imagen: Image.Image) -> np.ndarray:
    """ Calcula la silueta, ampliada TOLERANCIA_PX, tomando las esquinas como color de fondo.

    Args:
        * imagen: render en RGB

    Returns:
        * np.ndarray: matriz booleana (alto, ancho), True sobre el objeto

    """
    rgb = np.asarray(imagen, dtype=np.int16)
    fondo = np.median(rgb[[0, 0, -1, -1], [0, -1, 0, -1]], axis=0)
    mascara = np.abs(rgb - fondo).max(axis=2) > UMBRAL_FONDO
    mascara = Image.fromarray(np.uint8(mascara) * 255)
    return np.asarray(mascara.filter(ImageFilter.MaxFilter(2 * TOLERANCIA_PX + 1))) > 0


seleccion = {tipo: elegir_objetos(tipo) for tipo in ("axis_sym", "plane_sym")}
for tipo, objetos in seleccion.items():
    for obj_id, error in objetos:
        print(f"{tipo:10s} {obj_id}  error angular = {error:.2f} grados")

## Puntos sobre las imágenes

Cada fila es un objeto. Las cruces son los puntos con la conversión que usó el pipeline. El título
indica la vista y los puntos que caen fuera de la imagen, que no se pueden dibujar.

In [ ]:
COLORES = ["#e6194b", "#0082c8", "#3cb44b", "#f58231"]

for tipo, objetos in seleccion.items():
    figura, ejes = plt.subplots(len(objetos), N_VIEWS, figsize=(2.8 * N_VIEWS, 3.2 * len(objetos)),
                                squeeze=False)
    for fila, (obj_id, error) in enumerate(objetos):
        carpeta, grupo = cargar_grupo(tipo, obj_id)
        for vista, eje in zip(grupo["images_sent"], ejes[fila]):
            imagen = Image.open(carpeta / vista["filename"]).convert("RGB")
            ancho, alto = imagen.size
            eje.imshow(imagen, extent=(0, ancho, alto, 0), interpolation="nearest")
            puntos = grupo["points_by_image"].get(str(vista["img_idx"]), [])
            fuera = 0
            for j, punto in enumerate(puntos):
                x, y = punto["x"] / 1000 * ancho, punto["y"] / 1000 * alto
                fuera += not (0 <= x <= ancho and 0 <= y <= alto)
                eje.plot(x, y, "+", color=COLORES[j % len(COLORES)], markersize=14,
                         markeredgewidth=2.5)
            eje.set_xlim(0, ancho)
            eje.set_ylim(alto, 0)
            eje.set_xticks([])
            eje.set_yticks([])
            eje.set_title(f"vista {vista['index']}: {len(puntos)} pts, {fuera} fuera", fontsize=9)
        ejes[fila][0].set_ylabel(f"{obj_id[:10]}\nerror {error:.1f} grados", fontsize=9)
    figura.suptitle(f"{tipo}: Qwen3-VL, {N_VIEWS} vistas, {SIZE}px, {LIGHTING}")
    figura.tight_layout()
    ruta = CARPETA_SALIDA / f"{tipo}_qwen3vl_{N_VIEWS}vistas.png"
    figura.savefig(ruta, dpi=150)
    plt.show()
    print(f"Guardado: {ruta}")

## Comparación de hipótesis de escala

`% sobre objeto` compara cuántos puntos caen sobre la silueta con tres interpretaciones de los
valores que devolvió el modelo:

- **0-1000 [x, y]:** la que usó el pipeline.
- **píxeles [x, y]:** el modelo habría respondido en píxeles de la imagen.
- **0-1000 [y, x]:** el modelo habría invertido los ejes.

Con pocos objetos es una señal, no una medición definitiva.

In [ ]:
for tipo, objetos in seleccion.items():
    for obj_id, _ in objetos:
        carpeta, grupo = cargar_grupo(tipo, obj_id)
        denominador = grupo["scale_denominator"]
        crudos, aciertos = [], np.zeros(3)
        for vista in grupo["images_sent"]:
            imagen = Image.open(carpeta / vista["filename"]).convert("RGB")
            ancho, alto = imagen.size
            mascara = mascara_objeto(imagen)
            for punto in grupo["points_by_image"].get(str(vista["img_idx"]), []):
                cx, cy = punto["x"] * denominador / 1000, punto["y"] * denominador / 1000
                crudos.append((cx, cy))
                hipotesis = [(cx / denominador * ancho, cy / denominador * alto), (cx, cy),
                             (cy / denominador * ancho, cx / denominador * alto)]
                for k, (x, y) in enumerate(hipotesis):
                    dentro = 0 <= x < ancho and 0 <= y < alto
                    aciertos[k] += dentro and mascara[int(y), int(x)]
        print(f"{tipo} {obj_id[:10]}  n_puntos = {len(crudos)}")
        if crudos:
            crudos = np.array(crudos)
            porcentaje = 100 * aciertos / len(crudos)
            print(f"  valores crudos: x {crudos[:, 0].min():.0f} a {crudos[:, 0].max():.0f}, "
                  f"y {crudos[:, 1].min():.0f} a {crudos[:, 1].max():.0f}")
            print(f"  % sobre objeto: 0-1000 [x, y] = {porcentaje[0]:.0f}, "
                  f"píxeles [x, y] = {porcentaje[1]:.0f}, 0-1000 [y, x] = {porcentaje[2]:.0f}")
        salida_cruda = grupo["raw_output"][0] if grupo["raw_output"] else ""
        print(f"  primera respuesta: {str(salida_cruda)[:160]!r}")

## Cómo interpretar

- **Escala correcta:** las cruces caen sobre el objeto y `0-1000 [x, y]` tiene el mayor porcentaje.
  Algún valor crudo debería superar 224, porque en píxeles no podría hacerlo.
- **Respuesta en píxeles:** las cruces se amontonan arriba a la izquierda, ningún valor crudo supera
  224 y gana `píxeles [x, y]`.
- **Ejes invertidos:** las cruces forman el patrón reflejado en la diagonal y gana `0-1000 [y, x]`.
- **Escala correcta, pero puntos mal elegidos:** las cruces caen sobre el objeto, aunque no en los
  polos o la costura pedidos. Eso indica un error del modelo, no de la conversión.
- **Colapso al centro:** las cruces caen en el centro de todas las vistas. La escala puede estar
  bien, pero el buen resultado planar se debería a la pose canónica.